In [4]:
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
import wandb

DATA_DIR = "/home/nagaraj/Garbage_classification_files/Garbage classification"
OUTPUT_DIR = "partA_results"
PROJECT = "Atri"
ENTITY = "cs24s023-iitm-ac-in"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DEFAULT_CFG = {
    "filters": [32, 64, 128, 256, 512],
    "kernel_sizes": [3, 3, 3, 3, 3],
    "activation": "relu",
    "dense_neurons": 512,
    "use_batchnorm": True,
    "dropout": 0.3,
    "augment": True,
    "lr": 1e-3,
    "epochs": 20,
    "batch_size": 16,
    "input_size": 128,
    "num_classes": 6,
    "seed": 42,
    "patience": 5
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def approximate_flops_conv_linear(model, input_size=(3, 128, 128)):
    macs, in_ch = 0, input_size[0]
    H, W = input_size[1], input_size[2]
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            k_h, k_w = m.kernel_size
            s_h, s_w = m.stride
            p_h, p_w = m.padding
            H = (H + 2*p_h - k_h) // s_h + 1
            W = (W + 2*p_w - k_w) // s_w + 1
            macs += m.out_channels * H * W * (in_ch * k_h * k_w)
            in_ch = m.out_channels
        elif isinstance(m, nn.Linear):
            macs += m.in_features * m.out_features
    return macs

class FlexibleCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        filters, kernels = cfg["filters"], cfg["kernel_sizes"]
        act, use_bn = cfg["activation"].lower(), cfg["use_batchnorm"]
        layers, in_ch = [], 3
        for i in range(5):
            out_ch, k = filters[i], kernels[i]
            block = [nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=k//2)]
            if use_bn:
                block.append(nn.BatchNorm2d(out_ch))
            block.append(self._get_activation(act))
            block.append(nn.MaxPool2d(2))
            layers.append(nn.Sequential(*block))
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        self.adaptive = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(filters[-1], cfg["dense_neurons"])
        self.fc2 = nn.Linear(cfg["dense_neurons"], cfg["num_classes"])
        self.dropout = nn.Dropout(cfg["dropout"]) if cfg["dropout"] > 0 else None
        self._act_name = act

    def _get_activation(self, name):
        if name == "relu":
            return nn.ReLU(inplace=False)
        if name == "gelu":
            return nn.GELU()
        if name in ("silu", "swish"):
            return nn.SiLU()
        if name == "mish":
            return nn.Mish()
        return nn.ReLU(inplace=False)

    def forward(self, x):
        x = self.conv(x)
        x = self.adaptive(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self._get_activation(self._act_name)(x)
        if self.dropout:
            x = self.dropout(x)
        return self.fc2(x)

class GuidedBackpropSimple:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.hooks = []
        for module in self.model.modules():
            if isinstance(module, nn.ReLU):
                try:
                    self.hooks.append(module.register_full_backward_hook(self.relu_hook))
                except Exception:
                    pass
    def relu_hook(self, module, grad_in, grad_out):
        if grad_in is None:
            return None
        return tuple((g.clamp(min=0.0) if g is not None else None) for g in grad_in)
    def generate(self, img_tensor, device):
        if img_tensor.dim() == 3:
            img = img_tensor.unsqueeze(0).to(device)
        else:
            img = img_tensor.to(device)
        img.requires_grad = True
        out = self.model(img)
        target = out[0].max()  # keep it simple (not used for conv-layer guided mode)
        self.model.zero_grad()
        target.backward()
        grad = img.grad.detach().cpu()[0]
        return grad
    def close(self):
        for h in self.hooks:
            try:
                h.remove()
            except:
                pass

def save_guided(grad, path):
    g = grad.numpy().transpose(1,2,0)
    g = (g - g.min()) / (g.max() - g.min() + 1e-8)
    plt.imsave(path, np.clip(g,0,1))
    return path

def make_dataloaders(data_dir, input_size=128, batch_size=8, augment=True, seed=42):
    train_dir = os.path.join(data_dir, "train")
    test_dir = os.path.join(data_dir, "test")
    normalize = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    if augment:
        train_tf = transforms.Compose([
            transforms.RandomResizedCrop(input_size, scale=(0.8,1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2,0.2,0.2,0.1),
            transforms.ToTensor(),
            normalize
        ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize((input_size,input_size)),
            transforms.ToTensor(),
            normalize
        ])
    val_tf = transforms.Compose([transforms.Resize((input_size,input_size)), transforms.ToTensor(), normalize])
    train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
    test_ds = datasets.ImageFolder(test_dir, transform=val_tf)
    targets = np.array([s[1] for s in train_ds.samples])
    rng = np.random.RandomState(seed)
    train_idx, val_idx = [], []
    for c in np.unique(targets):
        idxs = np.where(targets==c)[0]
        rng.shuffle(idxs)
        n_train = max(1, int(0.8 * len(idxs)))
        train_idx += idxs[:n_train].tolist()
        val_idx += idxs[n_train:].tolist()
    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(Subset(train_ds, val_idx), batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds.classes

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_sum += float(loss.item()) * imgs.size(0)
        preds = outputs.argmax(1)
        correct += int((preds == labels).sum().item())
        total += labels.size(0)
    return loss_sum/total, correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss_sum += float(loss.item()) * imgs.size(0)
            preds = outputs.argmax(1)
            correct += int((preds == labels).sum().item())
            total += labels.size(0)
    return loss_sum/total, correct/total

def visualize_conv1_filters(model, out_path=os.path.join(OUTPUT_DIR, "conv1_filters.png")):
    conv1 = None
    # first conv layer in our conv sequential is conv[0][0]
    try:
        conv1 = model.conv[0][0]
    except Exception:
        for m in model.modules():
            if isinstance(m, nn.Conv2d):
                conv1 = m
                break
    if conv1 is None:
        return None
    w = conv1.weight.detach().cpu()
    grid = utils.make_grid(w, nrow=8, normalize=True, scale_each=True)
    utils.save_image(grid, out_path)
    return out_path

def plot_test_grid(model, loader, classes, device, out_path=os.path.join(OUTPUT_DIR, "test_grid_10x3.png")):
    model.eval()
    imgs_acc, labels_acc, preds_acc = [], [], []
    for imgs, labels in loader:
        imgs_acc.append(imgs)
        labels_acc.append(labels)
        with torch.no_grad():
            preds_acc.append(model(imgs.to(device)).argmax(1).cpu())
        if sum(x.size(0) for x in imgs_acc) >= 30:
            break
    if len(imgs_acc) == 0:
        return None
    imgs_all = torch.cat(imgs_acc)[:30]
    labels_all = torch.cat(labels_acc)[:30]
    preds_all = torch.cat(preds_acc)[:30]
    rows, cols = 10, 3
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    axes = axes.flatten()
    for i in range(len(imgs_all)):
        img = imgs_all[i].cpu().numpy().transpose(1,2,0)
        axes[i].imshow(np.clip(img, 0, 1))
        axes[i].set_title(f"P:{classes[int(preds_all[i])]} | T:{classes[int(labels_all[i])]}", fontsize=8)
        axes[i].axis("off")
    for j in range(len(imgs_all), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()
    return out_path

def guided_backprop_on_conv_layer(model, loader, device, num_maps=10):
    # find last conv layer (conv5)
    target_layer = None
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Conv2d):
            target_layer = m
            break
    if target_layer is None:
        return []
    activations = {}
    def forward_hook(module, inp, out):
        activations['out'] = out
    fh = target_layer.register_forward_hook(forward_hook)
    model.eval()
    imgs, labels = next(iter(loader))
    inp = imgs[0:1].to(device).requires_grad_(True)
    saved = []
    for ch in range(min(num_maps, activations.get('out', torch.empty(0)).shape[1] if 'out' in activations else num_maps)):
        # forward pass to populate activations
        activations.clear()
        out = model(inp)
        if 'out' not in activations:
            break
        act = activations['out']  # shape [1, C, H, W]
        if ch >= act.shape[1]:
            break
        # take mean of this channel's activation and backprop
        model.zero_grad()
        target = act[0, ch].mean()
        target.backward(retain_graph=True)
        grad = inp.grad.detach().cpu()[0]
        path = os.path.join(OUTPUT_DIR, f"guided_conv5_channel_{ch}.png")
        save_guided(grad, path)
        saved.append(path)
        inp.grad.zero_()
    fh.remove()
    return saved

sweep_config = {
    "method": "random",
    "metric": {"name": "val_acc", "goal": "maximize"},
    "parameters": {
        "filters": {"values": [[32,64,128,256,512],[64,64,64,64,64],[32,64,128,128,256]]},
        "activation": {"values": ["relu","gelu","silu","mish"]},
        "dense_neurons": {"values": [256,512]},
        "use_batchnorm": {"values": [True, False]},
        "dropout": {"values": [0.2,0.3]},
        "augment": {"values": [True, False]}
    }
}

def analyze_sweeps(results_csv):
    df = pd.read_csv(results_csv)
    plt.figure()
    plt.plot(df["val_acc"].values, marker="o")
    plt.title("Accuracy vs Runs")
    plt.xlabel("Run")
    plt.ylabel("Val Acc")
    p1 = os.path.join(OUTPUT_DIR, "acc_vs_runs.png")
    plt.savefig(p1); plt.close()
    from pandas.plotting import parallel_coordinates
    df_small = df[["activation","dropout","use_batchnorm","augment","val_acc"]].copy()
    df_small["val_acc"] = df_small["val_acc"].astype(float)
    plt.figure(figsize=(10,6))
    parallel_coordinates(df_small, "activation")
    p2 = os.path.join(OUTPUT_DIR, "parallel_coords.png")
    plt.title("Parallel Coordinates"); plt.savefig(p2); plt.close()
    corr = df.corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    sns.heatmap(corr, annot=True, cmap="coolwarm")
    p3 = os.path.join(OUTPUT_DIR, "correlation.png")
    plt.title("Correlation Matrix"); plt.savefig(p3); plt.close()
    corr.to_csv(os.path.join(OUTPUT_DIR,"correlation.csv"))
    return p1, p2, p3, corr

def write_readme_from_corr(corr):
    obs = []
    if "dropout" in corr.index and "val_acc" in corr.columns:
        if corr.loc["dropout","val_acc"] > 0:
            obs.append("Higher dropout correlated with higher validation accuracy.")
        else:
            obs.append("Higher dropout correlated with lower validation accuracy.")
    if "use_batchnorm" in corr.index and "val_acc" in corr.columns:
        if corr.loc["use_batchnorm","val_acc"] > 0:
            obs.append("Using BatchNorm correlated with higher validation accuracy.")
        else:
            obs.append("Using BatchNorm correlated with lower validation accuracy.")
    with open(os.path.join(OUTPUT_DIR, "README_PartA.md"), "w") as f:
        f.write("# Part A — Training CNN from Scratch\n\n")
        f.write("## Files\n")
        f.write("- conv1_filters.png: first-layer filter visualizations\n")
        f.write("- test_grid_10x3.png: sample test images with predictions\n")
        f.write("- guided_conv5_channel_*.png: guided backprop saliency for conv5 channels\n")
        f.write("- acc_vs_runs.png, parallel_coords.png, correlation.png: sweep analysis\n\n")
        f.write("## Observations (auto-generated)\n")
        for o in obs:
            f.write(f"- {o}\n")

def run_training(cfg_override=None):
    cfg = DEFAULT_CFG.copy()
    if cfg_override:
        cfg.update(cfg_override)
    set_seed(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    wandb.init(project=PROJECT, entity=ENTITY, config=cfg)
    train_loader, val_loader, test_loader, classes = make_dataloaders(DATA_DIR, cfg["input_size"], cfg["batch_size"], augment=cfg["augment"], seed=cfg["seed"])
    model = FlexibleCNN(cfg).to(device)
    params = count_parameters(model)
    macs = approximate_flops_conv_linear(model, (3, cfg["input_size"], cfg["input_size"]))
    wandb.config.update({"params_count": params, "approx_macs": macs}, allow_val_change=True)
    wandb.summary["params_count"] = params
    wandb.summary["approx_macs"] = macs
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    best_val, best_epoch, wait = 0.0, 0, 0
    for epoch in range(1, cfg["epochs"]+1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_acc)
        wandb.log({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": val_loss, "val_acc": val_acc})
        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
            wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
    wandb.log({"test_loss": test_loss, "test_acc": test_acc})
    wandb.summary["best_val_acc"] = best_val
    wandb.summary["best_epoch"] = best_epoch
    wandb.summary["test_acc"] = test_acc
    visualize_conv1_filters(model)
    plot_test_grid(model, test_loader, classes, device)
    guided_paths = guided_backprop_on_conv_layer(model, test_loader, device, num_maps=10)
    wandb.finish()
    return {
        "best_val": float(best_val),
        "best_epoch": int(best_epoch),
        "test_acc": float(test_acc),
        "params": params,
        "macs": macs,
        "guided_paths": guided_paths
    }

if __name__ == "__main__":
    if not os.path.isdir(DATA_DIR):
        raise RuntimeError(f"DATA_DIR not found: {DATA_DIR}")
    mode = "single"
    if mode == "single":
        summary = run_training()
        print("Run summary:", summary)
    else:
        sweep_id = wandb.sweep(sweep_config, project=PROJECT, entity=ENTITY)
        wandb.agent(sweep_id, function=lambda: run_training(), count=10)
        

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁
test_loss,▁
train_acc,▁▂▃▃▄▄▅▅▅▅▆▆▆▇▆▇▆▇██
train_loss,█▇▆▆▅▅▅▄▄▄▄▄▃▃▃▂▃▁▁▁
val_acc,▂▂▁▃▂▄▄▅▄▄▅▆▆▆▅▆▆█▇▇
val_loss,███▆▇▅▅▄▆▄▅▄▃▃▄▂▄▁▂▂
approx_macs,25683561472
best_epoch,18
best_val_acc,0.70617
epoch,20


Run summary: {'best_val': 0.7061728395061728, 'best_epoch': 18, 'test_acc': 0.65748031496063, 'params': 1836294, 'macs': 25683561472, 'guided_paths': ['partA_results/guided_conv5_channel_0.png', 'partA_results/guided_conv5_channel_1.png', 'partA_results/guided_conv5_channel_2.png', 'partA_results/guided_conv5_channel_3.png', 'partA_results/guided_conv5_channel_4.png', 'partA_results/guided_conv5_channel_5.png', 'partA_results/guided_conv5_channel_6.png', 'partA_results/guided_conv5_channel_7.png', 'partA_results/guided_conv5_channel_8.png', 'partA_results/guided_conv5_channel_9.png']}
